In [3]:
# pip install shap
# pip install "numpy<2"
import pandas as pd
import numpy as np
import os
import gc
import time
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

def aplicar_shap_mortalidad_rf():
    """
    - Descripción: Entrena el modelo definitivo de Random Forest para MORTALIDAD usando TODAS las variables
                   sobre la cohorte histórica (2019-2023). Luego, extrae y grafica los valores SHAP
                   para identificar el Top 20 de variables con mayor impacto en el riesgo.
    """
    target_name = 'MORTALIDAD'
    dir_datos = "../../Datos/Datasets Finales"
    dir_resultados = "../../Resultados/Resultados (etapa 5)"
    os.makedirs(dir_resultados, exist_ok=True)

    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER', 'DIAS_ESTADIA'] 

    print("="*60)
    print(f"INICIANDO FASE 5: SHAP PARA {target_name} (RANDOM FOREST)")
    print("="*60)

    # 1. Cargar datos de entrenamiento (Histórico 2019-2023 para evitar sesgo del 2024)
    print("[1/4] Cargando cohorte histórica (2019-2023)...")
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco_2019_2023.csv"), low_memory=False)
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control_2019_2023.csv"), low_memory=False)

    # 2. Balancear la cohorte basal
    print("[2/4] Balanceando la cohorte basal...")
    n_onco = len(df_onco_train)
    df_control_sample = df_control_train.sample(n=n_onco, random_state=42)
    df_train_maestro = pd.concat([df_onco_train, df_control_sample], ignore_index=True)

    del df_onco_train, df_control_train, df_control_sample
    gc.collect()

    features = [col for col in df_train_maestro.columns if col not in cols_excluir]
    X_train = df_train_maestro[features]
    y_train = df_train_maestro[target_name]

    # 3. Entrenar el modelo con los hiperparámetros óptimos y TODAS las variables
    print("[3/4] Entrenando modelo definitivo Random Forest...")
    inicio_train = time.time()
    modelo_rf = RandomForestClassifier(
        n_estimators=200, 
        max_depth=None, 
        min_samples_split=10, 
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )
    modelo_rf.fit(X_train, y_train)
    print(f"      -> Entrenamiento completado en {round(time.time() - inicio_train, 1)} segundos.")

    # 4. Cálculo de Valores SHAP
    print("[4/4] Calculando valores SHAP (Muestra representativa de 20.000 pacientes)...")
    # Tomamos una muestra para evitar el colapso de la RAM, manteniendo la proporción balanceada
    X_shap_sample = X_train.sample(n=20000, random_state=42)
    
    inicio_shap = time.time()
    # TreeExplainer es un algoritmo hiper-optimizado para Random Forest y XGBoost
    explainer = shap.TreeExplainer(modelo_rf)
    shap_values = explainer.shap_values(X_shap_sample)
    print(f"      -> SHAP calculado en {round((time.time() - inicio_shap)/60, 2)} minutos.")

    # Para Random Forest en Scikit-Learn, shap_values es una lista de arreglos [clase_0, clase_1].
    # Extraemos el índice 1, que corresponde a la fuerza que empuja hacia el FALLECIMIENTO.
    shap_valores_mortalidad = shap_values[1] 

    # 5. Generar y guardar el gráfico Summary Plot (Top 20)
    print("Generando gráfico Summary Plot...")
    plt.figure(figsize=(12, 8)) # Ajusta el tamaño de la figura
    
    # max_display=20 cumple exactamente la instrucción del profesor
    shap.summary_plot(shap_valores_mortalidad, X_shap_sample, max_display=20, show=False)
    
    ruta_grafico = os.path.join(dir_resultados, f"SHAP_Summary_Plot_{target_name}.png")
    plt.tight_layout()
    plt.savefig(ruta_grafico, dpi=300, bbox_inches='tight') # Guarda en alta resolución para la tesis
    plt.close()

    print("="*60)
    print(f"¡ÉXITO! Gráfico SHAP guardado en: {ruta_grafico}")
    print("="*60)

# Ejecutar la función
# aplicar_shap_mortalidad_rf()

In [ ]:
aplicar_shap_mortalidad_rf()

INICIANDO FASE 5: SHAP PARA MORTALIDAD (RANDOM FOREST)
[1/4] Cargando cohorte histórica (2019-2023)...
[2/4] Balanceando la cohorte basal...
[3/4] Entrenando modelo definitivo Random Forest...
      -> Entrenamiento completado en 150.1 segundos.
[4/4] Calculando valores SHAP (Muestra representativa de 20.000 pacientes)...
